# PRAKTIKUM DATA MINING
## Tugas 2 — Analisis Mendalam Aturan yang Belum Dibahas

- **Mata Kuliah**: Praktikum Data Mining
- **Topik**: Evaluasi Metrik Lanjutan (Leverage & Conviction) dan Interpretasi Bisnis Aturan Asosiasi
- **Dataset**: 60 Transaksi Toko Kelontong (`transactions_arm_praktikum.csv`)
- **Referensi Modul**: *Modul Praktikum Data Mining — Bagian 5.2 (Tugas 2)*


---
## Persiapan Data & Pustaka
Memuat library, dataset transaksi toko kelontong, melakukan encoding one-hot, dan mengekstrak aturan asosiasi (`min_support = 0.15`, `min_confidence = 0.5`) agar notebook ini mandiri (*self-contained*).


In [1]:
import pandas as pd
import numpy as np
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

# Membaca dataset
df_raw = pd.read_csv('transactions_arm_praktikum.csv')
transactions = [x.split(', ') for x in df_raw['Items'].dropna()]

# Encoding One-Hot
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df = pd.DataFrame(te_ary, columns=te.columns_)

# Mining Apriori & Association Rules
freq_apriori = apriori(df, min_support=0.15, use_colnames=True)
rules = association_rules(freq_apriori, metric='confidence', min_threshold=0.5)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print('Dataset 60 transaksi berhasil dimuat dan di-encode.')
print(f'Total frequent itemsets : {len(freq_apriori)}')
print(f'Total aturan asosiasi   : {len(rules)} aturan')


Dataset 60 transaksi berhasil dimuat dan di-encode.
Total frequent itemsets : 17
Total aturan asosiasi   : 18 aturan


---
# TUGAS 2 — Analisis Mendalam Aturan yang Belum Dibahas

Pada tugas ini, dipilih 2 aturan dari `rules.csv` yang belum dibahas pada Bagian 4.6:
1. **Aturan 1**: $\{\text{Telur}\} \rightarrow \{\text{Mie Instan}\}$
2. **Aturan 2**: $\{\text{Roti}\} \rightarrow \{\text{Susu}\}$

### 2.1 Perhitungan Manual Leverage dan Conviction
Rumus:
- $\text{Leverage}(X \rightarrow Y) = \text{Support}(X \cup Y) - [\text{Support}(X) \times \text{Support}(Y)]$
- $\text{Conviction}(X \rightarrow Y) = \frac{1 - \text{Support}(Y)}{1 - \text{Confidence}(X \rightarrow Y)}$


In [2]:
# Ekstraksi metrik dari rules DataFrame
rule_telur_mie = rules[(rules['antecedents'] == frozenset({'Telur'})) & (rules['consequents'] == frozenset({'Mie Instan'}))].iloc[0]
rule_roti_susu = rules[(rules['antecedents'] == frozenset({'Roti'})) & (rules['consequents'] == frozenset({'Susu'}))].iloc[0]

def hitung_evaluasi_manual(sup_x, sup_y, sup_xy, conf):
    lift = conf / sup_y
    leverage = sup_xy - (sup_x * sup_y)
    conviction = (1 - sup_y) / (1 - conf) if conf < 1.0 else float('inf')
    return lift, leverage, conviction

print("=== ATURAN 1: {Telur} -> {Mie Instan} ===")
l1, lev1, conv1 = hitung_evaluasi_manual(
    rule_telur_mie['antecedent support'],
    rule_telur_mie['consequent support'],
    rule_telur_mie['support'],
    rule_telur_mie['confidence']
)
print(f"Support(Telur)            : {rule_telur_mie['antecedent support']:.3f}")
print(f"Support(Mie Instan)       : {rule_telur_mie['consequent support']:.3f}")
print(f"Support(Telur, Mie Instan): {rule_telur_mie['support']:.3f}")
print(f"Confidence                : {rule_telur_mie['confidence']:.3f}")
print(f"Lift                      : {l1:.3f} (mlxtend: {rule_telur_mie['lift']:.3f})")
print(f"Leverage                  : {lev1:.4f} (mlxtend: {rule_telur_mie['leverage']:.4f})")
print(f"Conviction                : {conv1:.3f} (mlxtend: {rule_telur_mie['conviction']:.3f})")

print("\n=== ATURAN 2: {Roti} -> {Susu} ===")
l2, lev2, conv2 = hitung_evaluasi_manual(
    rule_roti_susu['antecedent support'],
    rule_roti_susu['consequent support'],
    rule_roti_susu['support'],
    rule_roti_susu['confidence']
)
print(f"Support(Roti)             : {rule_roti_susu['antecedent support']:.3f}")
print(f"Support(Susu)             : {rule_roti_susu['consequent support']:.3f}")
print(f"Support(Roti, Susu)       : {rule_roti_susu['support']:.3f}")
print(f"Confidence                : {rule_roti_susu['confidence']:.3f}")
print(f"Lift                      : {l2:.3f} (mlxtend: {rule_roti_susu['lift']:.3f})")
print(f"Leverage                  : {lev2:.4f} (mlxtend: {rule_roti_susu['leverage']:.4f})")
print(f"Conviction                : {conv2:.3f} (mlxtend: {rule_roti_susu['conviction']:.3f})")


=== ATURAN 1: {Telur} -> {Mie Instan} ===
Support(Telur)            : 0.250
Support(Mie Instan)       : 0.300
Support(Telur, Mie Instan): 0.167
Confidence                : 0.667
Lift                      : 2.222 (mlxtend: 2.222)
Leverage                  : 0.0917 (mlxtend: 0.0917)
Conviction                : 2.100 (mlxtend: 2.100)

=== ATURAN 2: {Roti} -> {Susu} ===
Support(Roti)             : 0.300
Support(Susu)             : 0.317
Support(Roti, Susu)       : 0.217
Confidence                : 0.722
Lift                      : 2.281 (mlxtend: 2.281)
Leverage                  : 0.1217 (mlxtend: 0.1217)
Conviction                : 2.460 (mlxtend: 2.460)


### 2.2 Analisis dan Interpretasi Bisnis
1. **Aturan $\{\text{Telur}\} \rightarrow \{\text{Mie Instan}\}$**:
   - Nilai $Lift = 2.222$ dan $Leverage = 0.0917$ mengindikasikan bahwa pembeli telur memiliki kemungkinan 2,22 kali lebih besar untuk membeli mie instan dibandingkan rata-rata pelanggan umum. Nilai conviction sebesar 2,100 memperlihatkan ketergantungan yang cukup kuat.
   - **Rekomendasi Bisnis**: Penempatan rak pajangan telur segar di dekat lorong mie instan atau promo "Beli 5 bungkus Mie Instan Gratis/Diskon 1 Butir Telur".
2. **Aturan $\{\text{Roti}\} \rightarrow \{\text{Susu}\}$**:
   - Nilai $Lift = 2.281$, $Confidence = 72.2\%$, dan $Leverage = 0.1217$. Sebanyak 12,17% transaksi di atas ekspektasi kebetulan acak terjadi karena pembelian bersama roti dan susu. Conviction sebesar 2,460 menandakan probabilitas kesalahan aturan 2,46 kali lebih kecil daripada jika roti dan susu independen.
   - **Rekomendasi Bisnis**: Paket bundling "Kombo Sarapan Sehat" (Roti Tawar + Susu UHT) dengan potongan harga Rp2.000,- untuk mendorong kenaikan basket size toko.


---
### 2.3 Tabel Ringkasan Komparasi Seluruh Metrik Evaluasi
Membandingkan seluruh metrik evaluasi (Support, Confidence, Lift, Leverage, Conviction) untuk kedua aturan secara terstruktur.


In [3]:
# Membuat tabel komparasi metrik rapi
summary_data = [
    {
        'Aturan': '{Telur} -> {Mie Instan}',
        'Support': 0.167,
        'Confidence': 0.667,
        'Lift': 2.222,
        'Leverage': 0.0917,
        'Conviction': 2.100
    },
    {
        'Aturan': '{Roti} -> {Susu}',
        'Support': 0.217,
        'Confidence': 0.722,
        'Lift': 2.281,
        'Leverage': 0.1217,
        'Conviction': 2.460
    }
]
df_summary = pd.DataFrame(summary_data)
print(df_summary.to_string())


                    Aturan  Support  Confidence   Lift  Leverage  Conviction
0  {Telur} -> {Mie Instan}    0.167       0.667  2.222    0.0917       2.100
1        {Roti} -> {Susu}    0.217       0.722  2.281    0.1217       2.460
